# Topic 16 — DBSCAN (Density-Based Clustering)

> **Goal of this notebook:** understand density-based clustering — group points that are packed closely together, label sparse points as **noise**, and discover **arbitrarily shaped** clusters that K-Means cannot. We'll also choose `eps` with a k-distance plot.

**Contents**
1. Concept & intuition
2. The mechanics (core/border/noise, eps, min_samples)
3. Choosing eps (the k-distance plot)
4. The data: two moons
5. DBSCAN vs K-Means on non-globular data
6. Noise detection & the effect of eps
7. Pros, cons & when to use
8. Summary

## 1. Concept & Intuition

DBSCAN (Density-Based Spatial Clustering of Applications with Noise) defines a cluster as a **dense region of points** separated from other dense regions by sparse areas. Unlike K-Means it:
- **doesn't need `k`** — it discovers the number of clusters,
- finds clusters of **any shape** (not just blobs),
- explicitly labels low-density points as **noise/outliers**.

It grows a cluster by connecting points that are densely packed together.

## 2. The Mechanics

Two parameters control density:
- **eps (ε):** the neighbourhood radius around a point.
- **min_samples:** how many points must lie within ε for a region to count as dense.

Every point is then one of:
- **Core point:** has ≥ `min_samples` neighbours within ε.
- **Border point:** within ε of a core point, but not itself core.
- **Noise point:** neither — an outlier.

Clusters form by chaining **density-reachable** core points together (and attaching their border points). Noise stays unassigned (label `-1`).

## 3. Choosing eps (the k-distance plot)

A standard heuristic: for each point, find the distance to its `min_samples`-th nearest neighbour, sort these distances, and plot them. The curve has an **"elbow"** — the distance there is a good `eps` (it's where points start becoming sparse). `min_samples` is often set to ~2×dimensions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN, KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.rcParams['figure.figsize'] = (8, 5)
print('Libraries loaded.')

## 4. The Data: Two Moons

Two interleaving crescents — a textbook case where density-based clustering wins.

In [ ]:
X, _ = make_moons(n_samples=400, noise=0.07, random_state=42)
X = StandardScaler().fit_transform(X)
plt.scatter(X[:, 0], X[:, 1], s=12, color='gray')
plt.title('Two moons (non-globular)'); plt.show()

In [ ]:
# k-distance plot to choose eps (min_samples = 5)
min_samples = 5
nbrs = NearestNeighbors(n_neighbors=min_samples).fit(X)
dists, _ = nbrs.kneighbors(X)
kdist = np.sort(dists[:, -1])
plt.plot(kdist)
plt.axhline(0.25, color='r', ls='--', label='eps ~ 0.25 (elbow)')
plt.xlabel('points (sorted)'); plt.ylabel(f'{min_samples}-th NN distance')
plt.title('k-distance plot'); plt.legend(); plt.show()

## 5. DBSCAN vs K-Means on Non-Globular Data

K-Means must cut the moons with a straight boundary; DBSCAN follows their density.

In [ ]:
db = DBSCAN(eps=0.25, min_samples=min_samples).fit(X)
km = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(X[:, 0], X[:, 1], c=db.labels_, cmap='viridis', s=12)
axes[0].set_title('DBSCAN (follows the crescents)')
axes[1].scatter(X[:, 0], X[:, 1], c=km.labels_, cmap='viridis', s=12)
axes[1].set_title('K-Means (forced straight cut)')
plt.show()

n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print('DBSCAN found', n_clusters, 'clusters and',
      int((db.labels_ == -1).sum()), 'noise points')

## 6. Noise Detection & the Effect of eps

`eps` is the key knob: too small → everything is noise; too large → clusters merge into one.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, eps in zip(axes, [0.1, 0.25, 0.5]):
    lab = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X)
    nc = len(set(lab)) - (1 if -1 in lab else 0)
    noise = int((lab == -1).sum())
    ax.scatter(X[:, 0], X[:, 1], c=lab, cmap='viridis', s=10)
    ax.set_title(f'eps={eps}: {nc} clusters, {noise} noise')
plt.show()

## 7. Pros, Cons & When to Use

**Pros**
- **No need to specify the number of clusters.**
- Finds **arbitrarily shaped** clusters.
- **Robust to outliers** — labels them as noise rather than forcing them into clusters.

**Cons**
- Very sensitive to `eps` and `min_samples`.
- Struggles when clusters have **very different densities**.
- Degrades in high dimensions (distances lose meaning); needs scaling.

**When to use**
- Spatial / geographic data, arbitrary shapes, or when outlier rejection matters.
- When you don't know `k` and clusters aren't globular.

## 8. Summary

- DBSCAN clusters **dense regions** via `eps` and `min_samples`, classifying points as **core / border / noise**.
- It discovers the number of clusters itself, captures **non-globular** shapes, and isolates **outliers** — succeeding on the moons where K-Means failed.
- The **k-distance plot** gives a principled way to choose `eps`, the most important parameter.